# Analise arquivos raw contas-pagar.csv

In [1139]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [1140]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR =  os.path.join(DATA_DIR, 'raw')

In [1141]:
df = pd.read_csv(os.path.join(RAW_DIR, 'contas_pagar.csv'), sep=';')
df.shape

(404, 10)

## Analise exploratória

In [1142]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_titulo_pagar    404 non-null    str    
 1   id_fornecedor      397 non-null    str    
 2   data_emissao       404 non-null    str    
 3   data_vencimento    404 non-null    str    
 4   data_pagamento     253 non-null    str    
 5   categoria_despesa  396 non-null    str    
 6   valor_titulo       397 non-null    float64
 7   valor_pago         397 non-null    float64
 8   status             397 non-null    str    
 9   forma_pagamento    397 non-null    str    
dtypes: float64(2), str(8)
memory usage: 31.7 KB


In [1143]:
df.isna().sum()

id_titulo_pagar        0
id_fornecedor          7
data_emissao           0
data_vencimento        0
data_pagamento       151
categoria_despesa      8
valor_titulo           7
valor_pago             7
status                 7
forma_pagamento        7
dtype: int64

In [1144]:
df.head()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
0,CP000001,F0005,2026-05-01,2026-05-22,2026-05-25,Mercadorias,12165.76,12165.76,Pago,Boleto
1,CP000002,F0002,2026-03-02,2026-04-01,NaN,Tecnologia,16428.88,0.00,Em aberto,PIX
2,CP000003,F0014,2026-03-30,2026-04-13,2026-04-12,Aluguel,19536.09,19536.09,Pago,Boleto
3,CP000004,F0022,2026-05-02,2026-06-16,NaN,Marketing,13913.86,0.00,Em aberto,Transferência
4,CP000005,F0019,2026-05-25,2026-06-15,NaN,Aluguel,12418.04,0.00,Em aberto,Boleto


In [1145]:
df.tail()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
399,CP000266,F0024,2026-03-28,2026-04-25,2026/04/27,Mercadorias,15682.59,15682.59,Pago,Boleto
400,CP000066,F0021,2026-07-27,2026-08-26,2026-08-30,Serviços,12909.99,12909.99,Pago,Transferência
401,CP000121,F0017,2026-04-16,2026-05-31,2026-05-29,Serviços,3961.71,3961.71,Pago,Transferência
402,CP000133,F0024,2026-06-01,2026-06-15,2026-06-18,Frete,14956.20,14956.20,Pago,Transferência
403,CP000379,F0002,2026-08-18,2026-08-25,2026-08-24,NaN,11610.80,11610.80,Pago,PIX


In [1146]:
df.sample(5)

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
296,CP000297,F0011,2026-05-26,2026-06-02,NaN,Mercadorias,19759.26,0.00,Em aberto,Transferência
38,CP000039,F0011,2026-02-27,2026-03-27,NaN,Tecnologia,8657.96,0.00,em aberto,Transferência
131,CP000132,F0008,2026-02-27,2026-03-20,2026-03-19,Mercadorias,2534.88,2534.88,Pago,Transferência
270,CP000271,F0023,2026-04-30,2026-05-28,NaN,Tecnologia,6435.50,0.00,em aberto,Débito automático
256,CP000257,F0013,2026-04-18,2026-05-18,NaN,Energia,10067.77,0.00,Em aberto,Boleto


In [1147]:
df.isnull().sum()

id_titulo_pagar        0
id_fornecedor          7
data_emissao           0
data_vencimento        0
data_pagamento       151
categoria_despesa      8
valor_titulo           7
valor_pago             7
status                 7
forma_pagamento        7
dtype: int64

In [1148]:
q = '''SELECT id_titulo_pagar
            , id_fornecedor
            , data_pagamento
            , categoria_despesa
            , valor_titulo
            , valor_pago
            , status
            , forma_pagamento
        FROM df
        WHERE id_fornecedor isnull
            or data_pagamento isnull
            or categoria_despesa isnull
            or valor_titulo isnull
            or valor_pago isnull
            or status isnull
            or forma_pagamento isnull

'''
pysqldf(q)

,id_titulo_pagar,id_fornecedor,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
0,CP000002,F0002,NaN,Tecnologia,16428.88,0.0,Em aberto,PIX
1,CP000004,F0022,NaN,Marketing,13913.86,0.0,Em aberto,Transferência
2,CP000005,F0019,NaN,Aluguel,12418.04,0.0,Em aberto,Boleto
3,CP000006,F0004,NaN,Frete,18233.22,0.0,Em aberto,PIX
4,CP000007,F0021,NaN,Frete,4797.91,0.0,Vencido,Boleto
...,...,...,...,...,...,...,...,...
174,CP000395,F0009,NaN,Marketing,13656.33,0.0,Em aberto,Boleto
175,CP000396,F0009,NaN,Marketing,17787.90,0.0,Em aberto,Débito automático
176,CP000397,F0009,NaN,Serviços,1061.72,0.0,Em aberto,Transferência
177,CP000398,F0020,NaN,Tecnologia,15957.82,0.0,Vencido,Transferência


In [1149]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_titulo_pagar    404 non-null    str    
 1   id_fornecedor      397 non-null    str    
 2   data_emissao       404 non-null    str    
 3   data_vencimento    404 non-null    str    
 4   data_pagamento     253 non-null    str    
 5   categoria_despesa  396 non-null    str    
 6   valor_titulo       397 non-null    float64
 7   valor_pago         397 non-null    float64
 8   status             397 non-null    str    
 9   forma_pagamento    397 non-null    str    
dtypes: float64(2), str(8)
memory usage: 31.7 KB


## Tratamento de dados

In [1150]:
df_original = df.copy()

In [1151]:
df = df.fillna('NAO INFORMADO')

In [1152]:
# Remove espaços antes/depois dos nomes
df.columns = df.columns.str.strip()

In [1153]:
campos_texto = [
    "id_titulo_pagar",
    "id_fornecedor",
    "categoria_despesa",
    "status",
    "forma_pagamento"
]

In [1154]:
for campo in campos_texto:

    df[campo] = (
        df[campo]
        .astype("string")
        .str.strip()
    )

In [1155]:
campos_numericos = [
    "valor_titulo",
    "valor_pago"
]

In [1156]:
for campo in campos_numericos:
    df[campo] = (
        df[campo].replace('NAO INFORMADO', 0)
        .astype("float")
    )

In [1157]:
# campos datas

df['data_emissao'] = df['data_emissao'].replace('NAO INFORMADO', '1900-01-01', regex=True)
df['data_emissao'] = df['data_emissao'].replace('/', '-',  regex=True)

df['data_vencimento'] = df['data_vencimento'].replace('NAO INFORMADO', '1900-01-01', regex=True)
df['data_vencimento'] = df['data_vencimento'].replace('/', '-',  regex=True)

df['data_pagamento'] = df['data_pagamento'].replace('NAO INFORMADO', '1900-01-01', regex=True)
df['data_pagamento'] = df['data_pagamento'].replace('/', '-',  regex=True)



In [1158]:
df['data_emissao'] = pd.to_datetime(df['data_emissao'], format='mixed')

In [1159]:
df['data_vencimento'] = pd.to_datetime(df['data_vencimento'], format='mixed')

In [1160]:
df['data_pagamento'] = pd.to_datetime(df['data_pagamento'], format='mixed')

In [1161]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_titulo_pagar    404 non-null    string        
 1   id_fornecedor      404 non-null    string        
 2   data_emissao       404 non-null    datetime64[us]
 3   data_vencimento    404 non-null    datetime64[us]
 4   data_pagamento     404 non-null    datetime64[us]
 5   categoria_despesa  404 non-null    string        
 6   valor_titulo       404 non-null    float64       
 7   valor_pago         404 non-null    float64       
 8   status             404 non-null    string        
 9   forma_pagamento    404 non-null    string        
dtypes: datetime64[us](3), float64(2), string(5)
memory usage: 31.7 KB


In [1162]:
df.head()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
0,CP000001,F0005,2026-05-01,2026-05-22,2026-05-25,Mercadorias,12165.76,12165.76,Pago,Boleto
1,CP000002,F0002,2026-03-02,2026-04-01,1900-01-01,Tecnologia,16428.88,0.00,Em aberto,PIX
2,CP000003,F0014,2026-03-30,2026-04-13,2026-04-12,Aluguel,19536.09,19536.09,Pago,Boleto
3,CP000004,F0022,2026-05-02,2026-06-16,1900-01-01,Marketing,13913.86,0.00,Em aberto,Transferência
4,CP000005,F0019,2026-05-25,2026-06-15,1900-01-01,Aluguel,12418.04,0.00,Em aberto,Boleto


In [1163]:
df.tail()

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
399,CP000266,F0024,2026-03-28,2026-04-25,2026-04-27,Mercadorias,15682.59,15682.59,Pago,Boleto
400,CP000066,F0021,2026-07-27,2026-08-26,2026-08-30,Serviços,12909.99,12909.99,Pago,Transferência
401,CP000121,F0017,2026-04-16,2026-05-31,2026-05-29,Serviços,3961.71,3961.71,Pago,Transferência
402,CP000133,F0024,2026-06-01,2026-06-15,2026-06-18,Frete,14956.20,14956.20,Pago,Transferência
403,CP000379,F0002,2026-08-18,2026-08-25,2026-08-24,NAO INFORMADO,11610.80,11610.80,Pago,PIX


In [1164]:
df.sample(5)

,id_titulo_pagar,id_fornecedor,data_emissao,data_vencimento,data_pagamento,categoria_despesa,valor_titulo,valor_pago,status,forma_pagamento
198,CP000199,F0005,2026-04-12,2026-05-10,2026-05-10,Serviços,14942.50,14942.50,Pago,Boleto
91,CP000092,F0017,2026-06-06,2026-07-21,2026-07-26,Mercadorias,8810.25,8810.25,Pago,Débito automático
222,CP000223,F0023,2026-02-18,2026-04-04,1900-01-01,Serviços,16337.04,0.00,Em aberto,PIX
79,CP000080,F0010,2026-03-02,2026-04-16,2026-04-15,Mercadorias,8712.83,8712.83,Pago,Boleto
241,CP000242,F0004,2026-05-19,2026-07-03,2026-07-04,Frete,16410.54,16410.54,Pago,Débito automático


## Salvando dados Contas Pagar em Bronze

In [1165]:
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')

In [1166]:
df.to_csv(
    os.path.join(BRONZE_DIR, 'b_contas_pagar.csv')
    , index=False
    , date_format="%Y-%m-%d"
)